# خوشه‌بندی K-Means — نوت‌بوک همراه ارائه 🎯

**درس یادگیری ماشین** | استاد درس: دکتر محمدرضا سلطان آقائی | ارائه‌دهنده: حجت‌اله داوری | آبان ۱۴۰۵

این نوت‌بوک همراهِ ارائه «بررسی جامع روش خوشه‌بندی K-Means» است و همه‌ی دموهایی را که در اسلایدها دیدید، به‌صورت قابل اجرا در اختیارتان می‌گذارد.

**نحوه استفاده:** از منوی `Runtime` گزینه‌ی `Run all` را بزنید (یا هر سلول را با `Shift+Enter` اجرا کنید). هیچ نصب اضافه‌ای لازم نیست — همه‌چیز در Colab از قبل موجود است.

**ساختار:**
۱. داده مصنوعی و اولین اجرای K-Means
۲. مراحل الگوریتم را گام‌به‌گام ببینیم
۳. انتخاب K — روش آرنج (Elbow)
۴. ارزیابی کیفیت — امتیاز سیلوئت (Silhouette)
۵. داده واقعی: بخش‌بندی مشتریان فروشگاه
۶. کاربرد جذاب: کاهش رنگ تصویر
۷. تمرین‌های پیشنهادی

---
### 🛠 راهنمای استفاده
- **ابزار:** همهٔ کدها در **Google Colab** اجرا می‌شوند — بدون نصبِ پایتون یا کتابخانه؛ فقط مرورگر و اینترنت (اجرای هر سلول: **Shift+Enter** یا منوی Runtime ▸ Run all).
- **یادگیری پایتون، از صفر:** [W3Schools — Python Tutorial](https://www.w3schools.com/python/default.asp) — آموزش تعاملی و رایگان؛ مثال‌ها را همان‌جا اجرا و ویرایش کنید.
- **هر سلول:** توضیح فارسی در بالای سلول و کامنت‌های فارسی داخل کد آمده است؛ خروجی‌ها هم از قبل ذخیره شده‌اند.


## ۱) داده مصنوعی و اولین اجرای K-Means

ابتدا داده‌ی تمرینی می‌سازیم: ۳۰۰ نقطه در ۴ گروه طبیعی (خوشه). سپس الگوریتم `KMeans` را با `K=4` اجرا می‌کنیم.

نکته‌ی مهم: **هیچ برچسبی به الگوریتم نمی‌دهیم** — خوشه‌ها را خودش کشف می‌کند. (این تفاوت اصلی با KNN و درخت تصمیم است که داده‌ی برچسب‌دار می‌خواهند.)

In [ ]:
# K-Means روی دادهٔ مصنوعی با ۴ خوشهٔ طبیعی — همان دموی اسلاید
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# 300 samples in 4 natural groups
X, y_true = make_blobs(n_samples=300, centers=4,
                       cluster_std=0.8, random_state=42)

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)          # cluster label of each point

plt.figure(figsize=(6, 4.5))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=25)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='black', marker='X', s=200, label='Centroids')
plt.title('K-Means on synthetic data (K=4)')
plt.legend()
plt.show()

print('Inertia (WCSS) =', round(kmeans.inertia_, 1))
print('Iterations until convergence =', kmeans.n_iter_)

## ۲) مراحل الگوریتم را گام‌به‌گام ببینیم

الگوریتم K-Means فقط دو کار را تکرار می‌کند: **انتساب** هر نقطه به نزدیک‌ترین مرکز، و **به‌روزرسانی** مراکز به میانگین اعضا. در سلول زیر همین دو گام را دستی پیاده‌سازی کرده‌ایم تا لحظه‌به‌لحظه ببینید چه اتفاقی می‌افتد.

In [ ]:
# K-Means دستی و گام‌به‌گام: تصویرِ هر تکرار (انتخاب تصادفی ← انتساب ← جابه‌جایی ← همگرایی)
def manual_kmeans(X, k, seed=3, max_iter=10):
    """Step-by-step K-Means: returns a snapshot (centroids, labels) per iteration."""
    rng = np.random.default_rng(seed)
    C = X[rng.choice(len(X), k, replace=False)]      # 1) random init
    snaps = []
    for _ in range(max_iter):
        d = np.linalg.norm(X[:, None] - C[None], axis=2)
        labels = d.argmin(axis=1)                    # 2) assignment step
        snaps.append((C.copy(), labels.copy()))
        newC = np.array([X[labels == j].mean(axis=0) for j in range(k)])  # 3) update
        if np.allclose(newC, C, atol=1e-4):          # 4) convergence check
            break
        C = newC
    return snaps

Xs, _ = make_blobs(n_samples=180, centers=3, cluster_std=0.95, random_state=7)
snaps = manual_kmeans(Xs, k=3)
print(f'converged after {len(snaps)} iterations')

fig, axes = plt.subplots(1, len(snaps), figsize=(4 * len(snaps), 3.4))
for i, ax in enumerate(np.atleast_1d(axes)):
    C, lab = snaps[i]
    ax.scatter(Xs[:, 0], Xs[:, 1], s=12, c=lab, cmap='tab10')
    ax.scatter(C[:, 0], C[:, 1], c='black', marker='X', s=140)
    ax.set_title(f'iteration {i+1}'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## ۳) انتخاب K — روش آرنج (Elbow)

`K` را از قبل باید خودمان تعیین کنیم. با رسم **Inertia** بر حسب K، نقطه‌ای که کاهش شدید ناگهان آهسته می‌شود (شکل آرنج) بهترین K است.

In [ ]:
# روش آرنج (Elbow): WCSS برای K=۱..۱۰ — شکستگی در K=۴
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4.3))
plt.plot(range(1, 11), inertias, 'o-', color='#2563eb')
plt.axvline(x=4, color='red', ls='--')
plt.annotate('Elbow -> K = 4', xy=(4, inertias[3]), xytext=(5.5, 5000),
             arrowprops=dict(arrowstyle='->', color='red'), color='red')
plt.xlabel('K (number of clusters)'); plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method')
plt.show()

for k, v in zip(range(1, 6), inertias[:5]):
    print(f'K={k}: inertia = {v:,.0f}')

## ۴) ارزیابی کیفیت — امتیاز سیلوئت (Silhouette)

برای هر نقطه: `s = (b − a) / max(a, b)` که `a` میانگین فاصله تا اعضای خوشه‌ی خودش و `b` میانگین فاصله تا نزدیک‌ترین خوشه‌ی دیگر است. بازه‌ی `[−۱, +۱]`؛ هرچه به ۱ نزدیک‌تر، خوشه‌ی فشرده‌تر و مجزاتر.

In [ ]:
# امتیاز Silhouette برای K=۲..۱۰ — تأیید همان K=۴
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 11):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    scores[k] = silhouette_score(X, lab)

best_k = max(scores, key=scores.get)
plt.figure(figsize=(6, 4.3))
colors = ['#2563eb' if k == best_k else '#cbd5e1' for k in scores]
plt.bar(list(scores), list(scores.values()), color=colors)
plt.xlabel('K'); plt.ylabel('Mean silhouette score')
plt.title(f'Silhouette Score vs K (best: K={best_k})')
plt.show()

print('best K by silhouette =', best_k, '-> score =', round(scores[best_k], 3))
print('(same answer as the Elbow method!)')

## ۵) داده واقعی: بخش‌بندی مشتریان فروشگاه 🛍️

مجموعه‌داده‌ی **Mall Customers** شامل ۲۰۰ مشتری با ویژگی‌های سن، جنسیت، درآمد سالانه (هزار دلار) و امتیاز خرید (۰ تا ۱۰۰). مشتریان را بر اساس «درآمد × رفتار خرید» به ۵ گروه تقسیم می‌کنیم — دقیقاً همان کاری که تیم‌های بازاریابی انجام می‌دهند.

In [ ]:
# بارگذاری دیتاست واقعی مشتریان مرکز خرید (۲۰۰ نفر) از GitHub
import pandas as pd

url = ('https://raw.githubusercontent.com/kennedykwangari/'
       'Mall-Customer-Segmentation-Data/master/Mall_Customers.csv')
df = pd.read_csv(url)
print(df.shape)
df.head()

In [ ]:
# بخش‌بندی مشتریان با K=۵ (درآمد × امتیاز خرید) + پروفایل میانگین هر سگمنت
X2 = df[['Annual Income (k$)', 'Spending Score (1-100)']]

km5 = KMeans(n_clusters=5, n_init=10, random_state=42)
df['Segment'] = km5.fit_predict(X2)     # new column: segment id per customer

plt.figure(figsize=(6.4, 4.6))
plt.scatter(X2.iloc[:, 0], X2.iloc[:, 1], c=df['Segment'], cmap='tab10', s=28)
plt.scatter(km5.cluster_centers_[:, 0], km5.cluster_centers_[:, 1],
            c='black', marker='X', s=220, label='Centroids')
plt.xlabel('Annual Income (k$)'); plt.ylabel('Spending Score (1-100)')
plt.title('Mall Customers - Segments (K=5)')
plt.legend()
plt.show()

print('silhouette =', round(silhouette_score(X2, df['Segment']), 3))
df.groupby('Segment')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)

**تفسیر سگمنت‌ها** (با نگاه به میانگین درآمد و خرید هر خوشه):

| سگمنت | ویژگی | اقدام پیشنهادی بازاریابی |
|---|---|---|
| هدف (درآمد بالا، خرید بالا) | مشتریان وفادار و سودده | برنامه‌ی وفاداری و محصولات لوکس |
| بالقوه (درآمد بالا، خرید کم) | پول دارند ولی خرج نمی‌کنند | تخفیف اولین خرید / پیشنهاد هدفمند |
| استاندارد (متوسط، متوسط) | اکثریت مشتریان | کمپین‌های عمومی |
| احساسی (درآمد پایین، خرید بالا) | حساس به پیشنهاد | تخفیف‌های مقطعی |
| صرفه‌جو (پایین، پایین) | کم‌خرج | خبرنامه‌ی ارزان‌قیمت |

**تمرین:** ستون‌های `Age` و `Spending Score` را جایگزین کنید و ببینید خوشه‌ها چطور عوض می‌شوند!

## ۶) کاربرد جذاب: کاهش رنگ تصویر با K-Means 🖼️

هر پیکسل تصویر را یک نمونه‌ی سه‌بعدی `(R, G, B)` در نظر بگیرید. اگر پیکسل‌ها را به `K` خوشه تقسیم کنیم، هر پیکسل با رنگِ مرکز خوشه‌اش جایگزین می‌شود — یعنی بازسازی کل تصویر فقط با `K` رنگ! (پایه‌ی فشرده‌سازی تصویر و پالت‌های رنگی)

In [ ]:
# کاربرد جالب K-Means: فشرده‌سازی تصویر — کاهش رنگ‌ها به ۸ و ۳۲ رنگ
from sklearn.datasets import load_sample_image

img = load_sample_image('china.jpg') / 255.0      # 427x640 pixels
H, W, _ = img.shape
Xpix = img.reshape(-1, 3)                          # 273,280 samples x 3 features
print('pixels =', H * W)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].imshow(img); axes[0].set_title('Original')
for ax, k in zip(axes[1:], [8, 32]):
    kmq = KMeans(n_clusters=k, n_init=4, random_state=42).fit(Xpix)
    compressed = kmq.cluster_centers_[kmq.labels_].reshape(img.shape)
    ax.imshow(np.clip(compressed, 0, 1))
    ax.set_title(f'K = {k} colors')
for ax in axes: ax.set_axis_off()
plt.show()

n_unique = len(np.unique((img * 255).astype(np.uint8).reshape(-1, 3), axis=0))
print(f'original image has {n_unique:,} unique colors -> reduced to only 8 / 32!')

## ۷) جمع‌بندی و تمرین‌های پیشنهادی ✅

- K-Means ساده‌ترین الگوریتم **خوشه‌بندی** (یادگیری بدون نظارت) است: تکرارِ «انتساب نقاط» و «به‌روزرسانی مراکز» تا همگرایی.
- انتخاب K با **روش آرنج** و کیفیت خوشه‌ها با **امتیاز سیلوئت** سنجیده می‌شود (در مثال ما هر دو K=4 را تأیید کردند).
- مرجع مطالعه: فصل ۱۰ کتاب *داده‌کاوی: مفاهیم و تکنیک‌ها* (هان و کامبر).

**تمرین‌ها:**
1. در دموی مشتریان، `K` را به ۴ یا ۶ تغییر دهید و سیلوئت را مقایسه کنید.
2. در دموی تصویر، `K=2` و `K=128` را امتحان کنید — از چه K ای دیگر تفاوت محسوس نیست؟
3. (چالشی) با `sklearn.cluster.DBSCAN` همین داده‌ی مشتریان را خوشه‌بندی کنید و نتیجه را با K-Means مقایسه کنید.
4. (چالشی) به‌جای RGB از فضای رنگ `HSV` استفاده کنید و ببینید نتیجه بهتر می‌شود یا نه.